# 4.2.4 Training Practices

Gradient clipping, mini-batch SGD, early stopping — practical deep learning.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

def clip_grads(gs,maxn=1.):
    n=np.sqrt(sum((g**2).sum() for g in gs))
    return [g*maxn/(n+1e-8) for g in gs] if n>maxn else gs

# mini-batch early stopping demo
X,y=make_moons(600,noise=.3,random_state=42); Xs=StandardScaler().fit_transform(X)
Xt,Xv,yt,yv=train_test_split(Xs,y,test_size=.25,random_state=42)
print('Data shapes:',Xt.shape,Xv.shape)

In [ ]:
import matplotlib.pyplot as plt
sigmoid=lambda x:1/(1+np.exp(-np.clip(x,-500,500))); relu=lambda x:np.maximum(0,x); relu_d=lambda x:(x>0).astype(float)
rng=np.random.default_rng(42)
W1=rng.standard_normal((2,32))*np.sqrt(2/2); b1=np.zeros(32)
W2=rng.standard_normal((32,1))*np.sqrt(2/32); b2=np.zeros(1)
best,wait=np.inf,0; patience=15; lr=.05; tl,vl=[],[]

for ep in range(300):
    idx=rng.permutation(len(yt))
    for s in range(0,len(yt),32):
        sl=idx[s:s+32]; Xb,yb=Xt[sl],yt[sl]
        z1=Xb@W1+b1; a1=relu(z1); a2=sigmoid(a1@W2+b2); y2=yb.reshape(-1,1); n=len(yb)
        dz2=(a2-y2)/n; dz1=(dz2@W2.T)*relu_d(z1)
        gs=clip_grads([Xb.T@dz1,a1.T@dz2]); W1-=lr*gs[0]; W2-=lr*gs[1]
    a2t=sigmoid(relu(Xt@W1+b1)@W2+b2); p=np.clip(a2t,1e-7,1-1e-7)
    tl.append(-np.mean(yt.reshape(-1,1)*np.log(p)+(1-yt.reshape(-1,1))*np.log(1-p)))
    a2v=sigmoid(relu(Xv@W1+b1)@W2+b2); pv=np.clip(a2v,1e-7,1-1e-7)
    vlo=-np.mean(yv.reshape(-1,1)*np.log(pv)+(1-yv.reshape(-1,1))*np.log(1-pv)); vl.append(vlo)
    if vlo<best: best=vlo; wait=0
    else:
        wait+=1
        if wait>=patience: print(f'Early stop ep={ep+1}'); break

plt.plot(tl,label='train'); plt.plot(vl,label='val'); plt.legend(); plt.title('Early Stopping'); plt.show()